In [3]:
# Task 1: News Topic Classifier
!pip install -q transformers datasets evaluate gradio torch scikit-learn

import numpy as np
import evaluate
import gradio as gr
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, logging

# Suppress warnings for a clean, human-like output
logging.set_verbosity_error()

print("Step 1: Loading AG News Dataset...")
dataset = load_dataset("ag_news")
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print("Step 2: Initializing DistilBERT Model...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

print("Step 3: Tokenizing text data...")
tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_test = small_test_dataset.map(tokenize_function, batched=True)

# Metrics setup
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

print("Step 4: Configuring Training Parameters...")
# We sync save_strategy with eval_strategy to fix the ValueError
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",     # Evaluate every epoch
    save_strategy="epoch",     # SAVE every epoch (Fixes the error)
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Step 5: Starting Fine-Tuning...")
trainer.train()

print("\nStep 6: Final Evaluation...")
eval_results = trainer.evaluate()
print(f"Results: {eval_results}")
# Gradio Deployment
labels_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

def predict_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    outputs = model(**inputs)
    predictions = outputs.logits.argmax(-1).item()
    return labels_map[predictions]

demo = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=2, placeholder="Paste a news headline here..."),
    outputs="text",
    title="News Topic Classifier (BERT)",
    description="Fine-tuned BERT model for AG News classification."
)

print("\nStep 7: Launching Gradio Web UI...")
demo.launch(share=True)

Step 1: Loading AG News Dataset...
Step 2: Initializing DistilBERT Model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Step 3: Tokenizing text data...
Step 4: Configuring Training Parameters...
Step 5: Starting Fine-Tuning...
{'eval_loss': '0.4137', 'eval_accuracy': '0.87', 'eval_f1': '0.8699', 'eval_runtime': '7.462', 'eval_samples_per_second': '67.01', 'eval_steps_per_second': '4.289', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.3954', 'eval_accuracy': '0.884', 'eval_f1': '0.8836', 'eval_runtime': '7.693', 'eval_samples_per_second': '65', 'eval_steps_per_second': '4.16', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '224', 'train_samples_per_second': '17.86', 'train_steps_per_second': '1.116', 'train_loss': '0.502', 'epoch': '2'}

Step 6: Final Evaluation...
{'eval_loss': '0.3954', 'eval_accuracy': '0.884', 'eval_f1': '0.8836', 'eval_runtime': '7.507', 'eval_samples_per_second': '66.61', 'eval_steps_per_second': '4.263', 'epoch': '2'}
Results: {'eval_loss': 0.39538663625717163, 'eval_accuracy': 0.884, 'eval_f1': 0.883643331351312, 'eval_runtime': 7.507, 'eval_samples_per_second': 66.605, 'eval_steps_per_second': 4.263, 'epoch': 2.0}

Step 7: Launching Gradio Web UI...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://33a662f9312be42ac5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
# Task 2: End-to-End ML Pipeline (Telco Churn)
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

print("Loading Dataset...")
# Downloading a public Telco Churn dataset directly into pandas
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Preprocessing Data...")
# Clean 'TotalCharges' (has blank spaces for new customers)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(" ", np.nan))
df.dropna(inplace=True)

# Define Features and Target
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Identify column types for the pipeline
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not  in numeric_features]

print("Building scikit-learn Pipeline...")
# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Main Pipeline
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Hyperparameter Tuning with GridSearchCV
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20]
}

print("Training & Tuning Model (GridSearchCV)...")
grid_search = GridSearchCV(clf_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")

print("Evaluating Model...")
y_pred = grid_search.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("Exporting Pipeline...")
joblib.dump(grid_search.best_estimator_, 'telco_churn_pipeline.pkl')
print("Pipeline successfully saved as 'telco_churn_pipeline.pkl'. You can now deploy this to production!")

Loading Dataset...
Preprocessing Data...
Building scikit-learn Pipeline...
Training & Tuning Model (GridSearchCV)...
Best Parameters: {'classifier__max_depth': 10, 'classifier__n_estimators': 50}
Evaluating Model...
Accuracy: 0.7882

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

Exporting Pipeline...
Pipeline successfully saved as 'telco_churn_pipeline.pkl'. You can now deploy this to production!


In [5]:
# Task 5: Auto Tagging Support Tickets Using LLM
!pip install -q transformers pandas

import pandas as pd
from transformers import pipeline

print("Loading Zero-Shot LLM Pipeline...")
# We use BART-large-MNLI which is a state-of-the-art zero-shot classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define potential tags for our support tickets
candidate_tags = [
    "Billing & Payments",
    "Technical Support",
    "Account Access/Password",
    "Feature Request",
    "Refunds & Returns",
    "Shipping Issue"
]

# Generate a mock dataset of support tickets to avoid needing external CSV uploads
data = {
    "ticket_id": [101, 102, 103, 104, 105],
    "description": [
        "I was charged twice for my subscription this month. Please help.",
        "The app keeps crashing every time I try to upload a profile picture on my Android phone.",
        "I forgot my password and the reset link is not arriving in my email.",
        "It would be great if you could add a dark mode to the dashboard.",
        "My package was supposed to arrive yesterday but the tracking hasn't updated."
    ]
}
df = pd.DataFrame(data)

def get_top_3_tags(text):
    # LLM infers the likelihood of each tag
    result = classifier(text, candidate_tags, multi_label=False)
    # Extract the top 3 tags with their probabilities
    top_3_labels = result['labels'][:3]
    top_3_scores = result['scores'][:3]

    # Format nicely
    formatted_tags = [f"{label} ({score*100:.1f}%)" for label, score in zip(top_3_labels, top_3_scores)]
    return formatted_tags

print("Processing Support Tickets...")
# Apply the LLM to our dataset
df['Top_3_Predicted_Tags'] = df['description'].apply(get_top_3_tags)

# Display the results clearly
pd.set_option('display.max_colwidth', None)
print("\n--- Support Ticket Tagging Results ---")
for index, row in df.iterrows():
    print(f"\nTicket ID: {row['ticket_id']}")
    print(f"Query: {row['description']}")
    print(f"Top 3 Tags: {row['Top_3_Predicted_Tags']}")

print("\nTask Complete! Zero-Shot LLM successfully tagged all tickets.")

Loading Zero-Shot LLM Pipeline...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Processing Support Tickets...

--- Support Ticket Tagging Results ---

Ticket ID: 101
Query: I was charged twice for my subscription this month. Please help.
Top 3 Tags: ['Billing & Payments (35.0%)', 'Refunds & Returns (18.0%)', 'Feature Request (15.8%)']

Ticket ID: 102
Query: The app keeps crashing every time I try to upload a profile picture on my Android phone.
Top 3 Tags: ['Technical Support (38.0%)', 'Feature Request (28.1%)', 'Refunds & Returns (12.1%)']

Ticket ID: 103
Query: I forgot my password and the reset link is not arriving in my email.
Top 3 Tags: ['Account Access/Password (56.8%)', 'Refunds & Returns (11.8%)', 'Technical Support (10.1%)']

Ticket ID: 104
Query: It would be great if you could add a dark mode to the dashboard.
Top 3 Tags: ['Feature Request (88.1%)', 'Refunds & Returns (2.8%)', 'Billing & Payments (2.4%)']

Ticket ID: 105
Query: My package was supposed to arrive yesterday but the tracking hasn't updated.
Top 3 Tags: ['Shipping Issue (72.6%)', 'Refunds & 